# Fine-tune PP-DocLayout_plus-L on Kaggle

Notebook này được thiết kế cho cách dùng `Run All` trên Kaggle.

Mục tiêu:

- fine-tune `PP-DocLayout_plus-L` trên Kaggle
- lấy checkpoint train về local
- export model sẽ làm ở local, không làm trong notebook này

Các file cần tải về sau khi train:

- `output/<run_name>/best_model/best_model.pdparams`
- `output/<run_name>/config.yaml`


### Cách dùng

1. Gắn Kaggle dataset chứa ảnh + annotation COCO
2. Gắn Kaggle dataset hoặc model input chứa pretrained weight `PP-DocLayout_plus-L_pretrained.pdparams`
3. Chỉ sửa các biến trong cell `Configuration` bên dưới
4. Bấm `Run All`


In [ ]:
from pathlib import Path

# ------------------------------
# Configuration
# ------------------------------
ROOT_DIR      = Path("/kaggle/working/PaddleX")
DATASET_DIR   = Path("/kaggle/input/my-dataset/dataset")
PRETRAIN_PATH = Path("/kaggle/input/my-pretrained/PP-DocLayout_plus-L_pretrained.pdparams")
PRETRAIN_URL  = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-DocLayout_plus-L_pretrained.pdparams"
PRETRAIN_PATH = Path("/kaggle/working/pretrained/PP-DocLayout_plus-L_pretrained.pdparams")
OUTPUT_DIR    = Path("/kaggle/working/output/pp_doclayout_plus_l")

NUM_CLASSES   = 3
DEVICE        = "gpu:0"
EPOCHS        = 80
BATCH_SIZE    = 1
LEARNING_RATE = 0.0001
WARMUP_STEPS  = 200
EVAL_INTERVAL = 10
LOG_INTERVAL  = 10

print("ROOT_DIR      =", ROOT_DIR)
print("DATASET_DIR   =", DATASET_DIR)
print("PRETRAIN_PATH =", PRETRAIN_PATH)
print("PRETRAIN_URL  =", PRETRAIN_URL)
print("PRETRAIN_PATH =", PRETRAIN_PATH)
print("OUTPUT_DIR    =", OUTPUT_DIR)
print("NUM_CLASSES   =", NUM_CLASSES)
print("DEVICE        =", DEVICE)


## 1. Validate Inputs

Cell này fail sớm nếu dataset chưa được mount đúng vào Kaggle input.

Với pretrained weight, notebook hỗ trợ 2 trường hợp:

- đã có sẵn trong Kaggle Input
- chưa có thì tự tải về `/kaggle/working/pretrained/`


In [ ]:
from pathlib import Path
import urllib.request

annotations_dir = DATASET_DIR / "annotations"
images_dir = DATASET_DIR / "images"
required_dataset_paths = [
    (DATASET_DIR, "Dataset directory"),
    (annotations_dir, "annotations directory"),
    (annotations_dir / "instance_train.json", "instance_train.json"),
    (annotations_dir / "instance_valid.json", "instance_valid.json"),
    (images_dir, "images directory"),
    (images_dir / "train", "images/train directory"),
    (images_dir / "valid", "images/valid directory"),
]

missing = []
for path, desc in required_dataset_paths:
    if not path.exists():
        missing.append(f"{desc} not found: {path}")

optional_test_paths = [
    (annotations_dir / "instance_test.json", "instance_test.json"),
    (images_dir / "test", "images/test directory"),
]
test_exists = [path.exists() for path, _ in optional_test_paths]
if any(test_exists) and not all(test_exists):
    for (path, desc), exists in zip(optional_test_paths, test_exists):
        if not exists:
            missing.append(f"{desc} not found: {path}")

if missing:
    raise FileNotFoundError("\n".join(missing))

split_image_counts = {}
for split in ("train", "valid", "test"):
    split_dir = images_dir / split
    if split_dir.exists():
        split_image_counts[split] = len([p for p in split_dir.rglob("*") if p.is_file()])

if split_image_counts.get("train", 0) == 0:
    raise FileNotFoundError(f"No images found under: {images_dir / 'train'}")
if split_image_counts.get("valid", 0) == 0:
    raise FileNotFoundError(f"No images found under: {images_dir / 'valid'}")

image_files = [p for p in images_dir.rglob("*") if p.is_file()]

if PRETRAIN_PATH.exists():
    FINAL_PRETRAIN_PATH = PRETRAIN_PATH
    print("Using pretrained weight from Kaggle Input:", FINAL_PRETRAIN_PATH)
else:
    PRETRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not PRETRAIN_PATH.exists():
        print("Pretrained weight not found in Input. Downloading from official URL...")
        urllib.request.urlretrieve(PRETRAIN_URL, PRETRAIN_PATH)
    FINAL_PRETRAIN_PATH = PRETRAIN_PATH
    print("Using downloaded pretrained weight:", FINAL_PRETRAIN_PATH)

print("Input validation passed.")
print("Train annotation:", annotations_dir / "instance_train.json")
print("Valid annotation:", annotations_dir / "instance_valid.json")
if (annotations_dir / "instance_test.json").exists():
    print("Test annotation: ", annotations_dir / "instance_test.json")
print("Images found:    ", len(image_files))
print("Split counts:    ", split_image_counts)
print("Final pretrained:", FINAL_PRETRAIN_PATH)


## 2. Clone PaddleX

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working

if [ ! -d /kaggle/working/PaddleX ]; then
  git clone https://github.com/PaddlePaddle/PaddleX.git
fi

cd /kaggle/working/PaddleX
git rev-parse HEAD


## 3. Install Environment

Stack này bám theo cấu hình đã dùng ổn trên Kaggle:

- `paddlepaddle-gpu==3.3.0`
- `pip install -e ".[base]"`
- `paddlex --install PaddleDetection -y`
- `setuptools<81`


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/PaddleX

python -m pip install --upgrade pip setuptools wheel
python -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
python -m pip install -e ".[base]"
paddlex --install PaddleDetection -y
python -m pip install "setuptools<81"


## 4. Check Dataset with PaddleX

Bước này phát hiện lỗi format COCO hoặc sai cấu trúc dữ liệu trước khi train.


In [ ]:
import os
import subprocess

env = os.environ.copy()
env.update({
    "HOME": "/kaggle/working/.home",
    "PADDLE_PDX_CACHE_HOME": "/kaggle/working/.paddlex_cache",
    "MPLCONFIGDIR": "/kaggle/working/.matplotlib",
    "PADDLE_PDX_LOCAL_FONT_FILE_PATH": "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
})

for d in [env["HOME"], env["PADDLE_PDX_CACHE_HOME"], env["MPLCONFIGDIR"], str(OUTPUT_DIR)]:
    Path(d).mkdir(parents=True, exist_ok=True)

cmd = [
    "python",
    str(ROOT_DIR / "main.py"),
    "-c", str(ROOT_DIR / "paddlex/configs/modules/layout_detection/PP-DocLayout_plus-L.yaml"),
    "-o", "Global.mode=check_dataset",
    "-o", f"Global.device={DEVICE}",
    "-o", f"Global.dataset_dir={DATASET_DIR}",
]

print("Running:")
print(" ".join(cmd))
subprocess.run(cmd, env=env, cwd=ROOT_DIR, check=True)


## 5. Train

Cell này bắt đầu fine-tune.

Mục tiêu output sau khi chạy xong:

- `OUTPUT_DIR / best_model / best_model.pdparams`
- `OUTPUT_DIR / config.yaml`


In [ ]:
train_cmd = [
    "python",
    str(ROOT_DIR / "main.py"),
    "-c", str(ROOT_DIR / "paddlex/configs/modules/layout_detection/PP-DocLayout_plus-L.yaml"),
    "-o", "Global.mode=train",
    "-o", f"Global.device={DEVICE}",
    "-o", f"Global.dataset_dir={DATASET_DIR}",
    "-o", f"Global.output={OUTPUT_DIR}",
    "-o", f"Train.num_classes={NUM_CLASSES}",
    "-o", f"Train.epochs_iters={EPOCHS}",
    "-o", f"Train.batch_size={BATCH_SIZE}",
    "-o", f"Train.learning_rate={LEARNING_RATE}",
    "-o", f"Train.warmup_steps={WARMUP_STEPS}",
    "-o", f"Train.eval_interval={EVAL_INTERVAL}",
    "-o", f"Train.log_interval={LOG_INTERVAL}",
    "-o", f"Train.pretrain_weight_path={FINAL_PRETRAIN_PATH}",
]

print("Running:")
print(" ".join(train_cmd))
subprocess.run(train_cmd, env=env, cwd=ROOT_DIR, check=True)


## 6. Verify Outputs

Bước này xác nhận có đủ file cần tải về local.


In [ ]:
expected_files = [
    OUTPUT_DIR / "config.yaml",
    OUTPUT_DIR / "best_model" / "best_model.pdparams",
]

missing_outputs = [str(p) for p in expected_files if not p.exists()]
if missing_outputs:
    raise FileNotFoundError("Missing expected output files:\n" + "\n".join(missing_outputs))

print("Expected output files found.")
for p in expected_files:
    print("-", p)


In [ ]:
%%bash
set -euo pipefail
echo "== output tree =="
find /kaggle/working/output -maxdepth 3 | sort


## 7. Zip Output For Download

Khuyến nghị tải cả thư mục output của run train, không chỉ riêng file weight.


In [ ]:
import zipfile

zip_path = Path("/kaggle/working/pp_doclayout_plus_l_output.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(OUTPUT_DIR.parent))

print("Saved zip to:", zip_path)


## 8. What To Download Back To Local

Tối thiểu:

```text
output/<run_name>/best_model/best_model.pdparams
output/<run_name>/config.yaml
```

Khuyến nghị:

- tải cả file zip vừa tạo
- export model sẽ làm ở local
- không dựa mặc định vào thư mục `inference/` export trên Kaggle để chạy local
